# Name That LLM：可提交基线

使用词和字符 TF-IDF 加线性 SVM 识别 18 类模型。按 `prompt_id` 做五折验证，评估 Balanced Accuracy；随后用全部训练数据拟合并输出 `submission.parquet`。

在 Kaggle Notebook 中添加本比赛数据，运行全部单元格，保存版本并使用 **Submit to Competition**。不要只上传本地生成的 parquet；本比赛要求代码提交。

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import balanced_accuracy_score, recall_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.svm import LinearSVC

KAGGLE_DATA = Path('/kaggle/input/competitions/name-that-llm')
LOCAL_DATA = Path('data')
if not LOCAL_DATA.exists():
    LOCAL_DATA = Path.cwd().parent / 'data'
DATA_DIR = KAGGLE_DATA if (KAGGLE_DATA / 'train.csv').exists() else LOCAL_DATA
OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
N_SPLITS = 5
print(f'Data: {DATA_DIR.resolve()}')
print(f'Output: {OUTPUT_DIR.resolve()}')

In [ ]:
train_path = DATA_DIR / 'train.csv'
test_path = DATA_DIR / 'test.csv'
if not train_path.exists() or not test_path.exists():
    raise FileNotFoundError(f'Expected train.csv and test.csv in {DATA_DIR}')

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
required_train = {'id', 'prompt_id', 'text', 'model'}
required_test = {'id', 'text'}
if not required_train.issubset(train.columns):
    raise ValueError(f'Train columns missing: {sorted(required_train - set(train.columns))}')
if not required_test.issubset(test.columns):
    raise ValueError(f'Test columns missing: {sorted(required_test - set(test.columns))}')
if train[list(required_train)].isna().any().any() or test[list(required_test)].isna().any().any():
    raise ValueError('Required columns contain null values')
if train['id'].duplicated().any() or test['id'].duplicated().any():
    raise ValueError('IDs must be unique within each file')
if (train['text'].astype(str).str.strip() == '').any() or (test['text'].astype(str).str.strip() == '').any():
    raise ValueError('Text cannot be empty')

labels = sorted(train['model'].unique().tolist())
if len(labels) != 18:
    raise ValueError(f'Expected 18 model labels, found {len(labels)}')
if train['prompt_id'].nunique() < N_SPLITS:
    raise ValueError(f'Need at least {N_SPLITS} distinct prompt_id values')

print(f'Train rows: {len(train):,}; test rows: {len(test):,}')
print(f'Prompts: {train["prompt_id"].nunique()}; classes: {len(labels)}')
print(train['model'].value_counts().sort_index().to_string())

In [ ]:
def make_model():
    return Pipeline([
        ('features', FeatureUnion([
            ('word', TfidfVectorizer(
                analyzer='word', ngram_range=(1, 2), min_df=2,
                max_features=40_000, sublinear_tf=True, dtype=np.float32,
            )),
            ('char', TfidfVectorizer(
                analyzer='char', ngram_range=(2, 5), min_df=2,
                max_features=80_000, sublinear_tf=True, dtype=np.float32,
            )),
        ])),
        ('classifier', LinearSVC(class_weight='balanced', max_iter=3_000)),
    ])

model = make_model()
cv = GroupKFold(n_splits=N_SPLITS)
fold_scores = []
oof_predictions = np.empty(len(train), dtype=object)

for fold, (fit_idx, valid_idx) in enumerate(
    cv.split(train['text'], train['model'], groups=train['prompt_id']), start=1
):
    fit_prompts = set(train.iloc[fit_idx]['prompt_id'])
    valid_prompts = set(train.iloc[valid_idx]['prompt_id'])
    assert fit_prompts.isdisjoint(valid_prompts)
    fold_model = clone(model)
    fold_model.fit(train.iloc[fit_idx]['text'].astype(str), train.iloc[fit_idx]['model'])
    pred = fold_model.predict(train.iloc[valid_idx]['text'].astype(str))
    oof_predictions[valid_idx] = pred
    score = balanced_accuracy_score(train.iloc[valid_idx]['model'], pred)
    fold_scores.append({'fold': fold, 'n_valid': len(valid_idx), 'balanced_accuracy': score})
    print(f'Fold {fold}: {score:.4f} ({len(valid_idx)} rows)')

fold_results = pd.DataFrame(fold_scores)
overall_score = balanced_accuracy_score(train['model'], oof_predictions)
class_results = pd.DataFrame({
    'model': labels,
    'recall': recall_score(train['model'], oof_predictions, labels=labels, average=None, zero_division=0),
})
print(f'Mean fold score: {fold_results["balanced_accuracy"].mean():.4f}')
print(f'Out-of-fold score: {overall_score:.4f}')
display(class_results.sort_values('recall').reset_index(drop=True))

fold_results.to_csv(OUTPUT_DIR / 'cv_results.csv', index=False)
class_results.to_csv(OUTPUT_DIR / 'class_recall.csv', index=False)

In [ ]:
final_model = make_model()
final_model.fit(train['text'].astype(str), train['model'])
predictions = final_model.predict(test['text'].astype(str))
submission = pd.DataFrame({'id': test['id'], 'model': predictions})

if len(submission) != len(test):
    raise ValueError('Submission row count does not match test')
if submission['id'].isna().any() or submission['model'].isna().any():
    raise ValueError('Submission contains null values')
if submission['id'].duplicated().any() or set(submission['id']) != set(test['id']):
    raise ValueError('Submission IDs do not match test IDs exactly')
if not set(submission['model']).issubset(set(labels)):
    raise ValueError('Submission contains an invalid model label')

submission_path = OUTPUT_DIR / 'submission.parquet'
submission.to_parquet(submission_path, index=False)
check = pd.read_parquet(submission_path)
if list(check.columns) != ['id', 'model'] or len(check) != len(test):
    raise ValueError('Saved parquet file failed verification')
print(f'Saved {submission_path} ({len(submission):,} rows)')
display(submission.head())